In [ ]:
import pathlib
import pickle
from distutils.command.bdist import bdist

from adaptive_latents import StimRegressor
from adaptive_latents.stim_designer import StimDesigner
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

rng = np.random.default_rng(0)

In [ ]:
with open(pathlib.Path('/home/jgould/Documents/neurips_2025/generated/') / 'optimization_history.pkl', 'rb') as fhan:
    sr = pickle.load(fhan)
sr: StimRegressor

In [ ]:
sr.stim_designer.log[-1].keys()

In [ ]:
sr.stim_designer.log[-1]['stim_reg'].history


In [ ]:
for l in sr.stim_designer.log:
    s = l['s']
    v = l['v']
    stim_reg = l['stim_reg']

    assert np.allclose(v.T @ v, np.eye(v.shape[1]))
    s_proj = v @ v.T @ s
    in_norm = np.linalg.norm(s_proj)**2
    out_norm = np.linalg.norm(s -s_proj)**2
    print( in_norm / out_norm)

In [ ]:
a_s = np.linspace(.01, .99, 2)
a_s_label = 'a1'
b_s = np.linspace(1,2, 2).astype(int)
b_s_label = 'nothing'
depth = 1

times_accumulator = []
objective_accumulator = []

with tqdm(total=len(a_s) * len(b_s) * depth * len(sr.stim_designer.log)) as pbar:
    for a in a_s:
        times_accumulator.append([])
        objective_accumulator.append([])
        for b in b_s:
            times_accumulator[-1].append([])
            objective_accumulator[-1].append([])

            for _ in range(depth):
                times_accumulator[-1][-1].append([])
                objective_accumulator[-1][-1].append([])
                sd = StimDesigner(
                    max_l0_norm=sr.stim_designer.max_l0_norm,
                    l0_norm_margin=sr.stim_designer.l0_norm_margin,
                    adaptive_starter_lam_1=True,
                    max_outer_loop_time_ms=1000, # or 20
                    max_inner_iters=100, # or 60
                    rng_seed=rng.integers(2**32),
                    should_log=True,
                    # a1=a
                )
                # sd.adam_learning_rate = 10**-2.333
                # sd.convergence_threshold = 10**-2.778


                for l in sr.stim_designer.log:
                    old_s = l['s']
                    v = l['v']

                    new_s = sd.design_stim(v)

                    s_proj = v @ v.T @ new_s
                    in_norm = np.linalg.norm(s_proj)**2
                    out_norm = np.linalg.norm(new_s -s_proj)**2
                    if not np.isclose(new_s, 0).all():
                        objective = in_norm / out_norm if out_norm > 0 else np.nan
                    else:
                        objective = 0

                    times_accumulator[-1][-1][-1].append(sd.log[-1]['time'])
                    objective_accumulator[-1][-1][-1].append(objective)
                    pbar.update(1)

times = np.array(times_accumulator)
objectives = np.array(objective_accumulator)
objectives[np.isnan(objectives)] = np.nanmax(objectives)

as_array, bs_array = np.meshgrid(b_s, a_s)


In [ ]:

fig, ax = plt.subplots()
im = ax.pcolormesh(bs_array, as_array, np.median(objectives, axis=(-2, -1)), shading='nearest')
fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_ylabel(f'b_s: {b_s_label}')
ax.set_xlabel(f'a_s: {a_s_label}')



In [ ]:
%matplotlib inline
fig, ax = plt.subplots()
im = ax.pcolormesh(bs_array, as_array, np.median(times, axis=(-2, -1)) * 1000, shading='nearest')
fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_ylabel(f'b_s: {b_s_label}')
ax.set_xlabel(f'a_s: {a_s_label}')



In [ ]:

all_loss_history  = []
all_lam_1_history  = []
all_s_history  = []

for l in sd.log:
    # 'time', 'v', 's', 'loss_history', 'lam_1_history', 'l0_history', 's_history'
    lam_1_history = l['lam_1_history']
    loss_history = l['loss_history']
    s = l['s']
    s_history = l['s_history']


    flat_loss_history = np.squeeze(np.hstack(loss_history))
    flat_lam_1_history = np.hstack([[lam_1] *len(loss_h) for lam_1, loss_h  in zip(lam_1_history, loss_history)])
    flat_s_history = np.vstack(s_history).T
    optimal_index = np.argmax(((flat_s_history / flat_s_history.max(axis=0)).T == s).all(axis=1))

    all_loss_history.append(flat_loss_history)
    all_lam_1_history.append(flat_lam_1_history)
    all_s_history.append(flat_s_history)

fig, axs = plt.subplots(nrows=3, sharex=True, figsize=(5, 10), height_ratios=[1,1,2])
axs[0].plot(flat_loss_history)
axs[1].plot(np.log(flat_lam_1_history))

axs[2].imshow(flat_s_history, aspect='auto', interpolation='none')

axis = axs[2].axis()
axs[2].set_xticks(list(axs[2].get_xticks()) + [optimal_index], labels=list(axs[2].get_xticklabels()) + [f'\n{optimal_index}'])
axs[2].axis(axis)

In [ ]:

fig, axs = plt.subplots(nrows=3, sharex=True, figsize=(5, 10), height_ratios=[1,1,2])
axs[0].plot(np.hstack(all_loss_history))
axs[1].plot(np.log(np.hstack(all_lam_1_history)))
axs[2].imshow(np.hstack(all_s_history), aspect='auto', interpolation='none')
#
# axis = axs[2].axis()
# axs[2].set_xticks(list(axs[2].get_xticks()) + [optimal_index], labels=list(axs[2].get_xticklabels()) + [f'\n{optimal_index}'])
# axs[2].axis(axis)


In [ ]:
plt.plot(lam_1_history)

In [ ]:

loss_history